# SvasaLens neural network v2

An improved neural network, built to work **across hospitals**, not just at the one it learned from.

**What is new compared with the first network:**
1. **Two training hospitals:** PhysioNet 2018 (Boston) and PSG-Audio (Athens), so it cannot rely on one hospital's equipment.
2. **Varied training examples:** each signal is randomly flipped, resized, given noise and slow drift, and shifted slightly in time. The sign flip matters for the device: when SvasaLens calibrates, the plus/minus direction of each sensor comes out at random.
3. **Chip-style input:** the network sees the same 8 Hz signals, through the same filter, that the chip uses.
4. **Fairer training:** it stops when the tuning patients' AUC stops improving, and the obstructive/central cut-off is set on the tuning patients only.

**Models compared:** your original rule; the network trained on PhysioNet only; trained on both hospitals; and both hospitals plus airflow (the thermistor).

**Tests:** each hospital's held-out test patients, all PSG-Audio patients (only for models that never trained on it, so a third hospital for your rule), and UCDDB (never used by any new model).

**Needs:** the PhysioNet 2018 notebook finished, and the PSG-Audio notebook at least partly done (it uses whatever PSG-Audio patients are ready; without any, it trains on PhysioNet only). For PSG-Audio, run section 7 of that notebook first, so the rule results exist.

**How to run:** Runtime > Run all. A GPU (Runtime > Change runtime type > T4 GPU) makes it faster, but CPU works. Results go to `My Drive > SvasaLens_NN_v2`.


## 1. Helper code

In [ ]:
# Helper code (from SvasaLens). You don't need to change anything here.
SVNN_PY = '"""SvasaLens neural network v2: chip-style input, two training hospitals, varied examples."""\nimport numpy as np\n\nFS = 8\nPRE, POST = 30, 60          # window: 30 s before to 60 s after the event start\nN = (PRE + POST) * FS\n\n\ndef chip_filter(x):\n    """The chip\'s own filter (high-pass 0.1 Hz then low-pass 0.5 Hz, run forwards in time like the chip)."""\n    from scipy.signal import lfilter\n    import chipref_v2 as C\n    y = np.asarray(x, np.float64)\n    for name in (\'HP1\', \'LP1\'):\n        b0, b1, b2, a1, a2 = C.COEF[name]\n        y = lfilter([b0, b1, b2], [1, a1, a2], y)\n    return y.astype(np.float32)\n\n\ndef flow_filter(x):\n    """Airflow at 8 Hz, filtered 0.1-1 Hz forwards in time (what the device could do with the thermistor)."""\n    from scipy.signal import butter, lfilter\n    b, a = butter(2, [0.1 / 4, 1.0 / 4], btype=\'band\')\n    return lfilter(b, a, np.nan_to_num(np.asarray(x, np.float64))).astype(np.float32)\n\n\ndef window(sigs, t0, dur):\n    """sigs: list of 8 Hz filtered signals. Returns len(sigs)+1 rows (signals + event marker), each signal\n    scaled by its size in the 30 s before the event, like the chip\'s normal-breathing baseline."""\n    out = np.zeros((len(sigs) + 1, N), np.float32)\n    a = int(round((t0 - PRE) * FS))\n    for k, x in enumerate(sigs):\n        lo, hi = max(0, a), min(len(x), a + N)\n        if hi <= lo: continue\n        w = np.zeros(N, np.float32); w[lo - a:hi - a] = x[lo:hi]\n        pre = w[:PRE * FS]\n        sc = np.sqrt(np.mean(pre ** 2))\n        if sc > 0: out[k] = np.clip(w / sc, -20, 20)\n    i0 = PRE * FS\n    out[-1, i0:min(N, i0 + int(round(dur * FS)))] = 1.0\n    return out\n\n\ndef augment(x, rng):\n    """Random changes so the network learns the breathing pattern, not one machine\'s signal:\n    sign flip and size change per signal (the device\'s sensor direction is random), noise,\n    slow drift, and a small shift in time."""\n    x = x.copy()\n    B, C, T = x.shape\n    sig = C - 1                                           # last row is the event marker\n    x[:, :sig] *= rng.choice([-1.0, 1.0], size=(B, sig, 1)).astype(np.float32)\n    x[:, :sig] *= np.exp(rng.uniform(np.log(0.6), np.log(1.6), size=(B, sig, 1))).astype(np.float32)\n    x[:, :sig] += rng.normal(0, 0.1, size=(B, sig, T)).astype(np.float32)\n    t = np.linspace(0, 1, T, dtype=np.float32)\n    x[:, :sig] += (rng.normal(0, 0.3, size=(B, sig, 1)) * np.sin(np.pi * t * rng.uniform(0.5, 2, size=(B, 1, 1)))).astype(np.float32)\n    s = rng.integers(-2 * FS, 2 * FS + 1, size=B)\n    for i in np.flatnonzero(s):\n        x[i, :sig] = np.roll(x[i, :sig], s[i], axis=-1)\n    return x\n'
SVEXP_PY = '"""SvasaLens experiments: extra features (airflow, SpO2) and a small neural network."""\nimport os, json, re\nimport numpy as np\n\nWIN_PRE, WIN_POST, NN_FS = 30, 60, 8          # NN window: 30 s before to 60 s after the event start, at 8 Hz\nRULE = (1.0999081134796143, 2.603668451309204, 5.435161113739014)\n\n\n# ---------- signal helpers ----------\ndef bandpass(x, fs, lo=0.1, hi=1.0):\n    from scipy.signal import butter, sosfiltfilt\n    x = np.nan_to_num(np.asarray(x, np.float64))\n    sos = butter(2, [lo / (fs / 2), hi / (fs / 2)], btype=\'band\', output=\'sos\')\n    return sosfiltfilt(sos, x).astype(np.float32)\n\n\ndef resample(x, rate, target):\n    from scipy.signal import resample_poly\n    from fractions import Fraction\n    fr = Fraction(int(round(target)), int(round(rate))).limit_denominator(1000)\n    return resample_poly(np.nan_to_num(np.asarray(x, np.float64)), fr.numerator, fr.denominator).astype(np.float32)\n\n\ndef clean_spo2(s):\n    s = np.asarray(s, np.float32).copy()\n    s[(s < 50) | (s > 100)] = np.nan\n    return s\n\n\ndef _seg(x, fs, a, b):\n    i0, i1 = max(0, int(a * fs)), min(len(x), int(b * fs))\n    return x[i0:i1] if i1 > i0 else x[:0]\n\n\n# ---------- hand-made features ----------\nFEATURES = [\'ln_effort\', \'rhythm\', \'ln_flow_ratio\', \'ln_flow_recovery\', \'desat\', \'nadir_delay\']\n\n\ndef event_features(effort, rhythm, flow_bp, spo2, fs, t0, dur):\n    """Effort and rhythm come from the chip\'s detector. Airflow and SpO2 features are new."""\n    f = {\'ln_effort\': np.log(max(effort, 1e-3)), \'rhythm\': rhythm}\n    base = _seg(flow_bp, fs, t0 - 60, t0 - 5)\n    base = np.sqrt(np.mean(base ** 2)) if len(base) > fs * 10 else np.nan\n    during = _seg(flow_bp, fs, t0, t0 + dur)\n    after = _seg(flow_bp, fs, t0 + dur, t0 + dur + 15)\n    ok = base == base and base > 0\n    f[\'ln_flow_ratio\'] = np.log(max(np.sqrt(np.mean(during ** 2)) / base, 1e-3)) if ok and len(during) else np.nan\n    f[\'ln_flow_recovery\'] = np.log(max(np.max(np.abs(after)) / (base * np.sqrt(2)), 1e-3)) if ok and len(after) else np.nan\n    sb = _seg(spo2, fs, t0 - 60, t0)\n    sw = _seg(spo2, fs, t0, t0 + dur + 40)\n    if len(sb) and len(sw) and np.isfinite(sb).sum() > fs * 10 and np.isfinite(sw).sum() > fs * 5:\n        f[\'desat\'] = float(np.nanmedian(sb) - np.nanmin(sw))\n        f[\'nadir_delay\'] = float(np.nanargmin(sw) / fs - dur)\n    else:\n        f[\'desat\'] = f[\'nadir_delay\'] = np.nan\n    return f\n\n\n# ---------- neural-network input window ----------\ndef event_window(chest, abd, flow, spo2, fs, t0, dur):\n    """4 signals + event mask, 90 s around the event at 8 Hz, each scaled by the 30 s before the event."""\n    n = (WIN_PRE + WIN_POST) * NN_FS\n    out = np.zeros((5, n), np.float32)\n    a = t0 - WIN_PRE\n    for k, x in enumerate((chest, abd, flow)):\n        w = _seg(x, fs, a, a + WIN_PRE + WIN_POST)\n        w = resample(w, fs, NN_FS)[:n] if len(w) else np.zeros(0, np.float32)\n        pre = w[:WIN_PRE * NN_FS]\n        sc = np.sqrt(np.mean((pre - pre.mean()) ** 2)) if len(pre) > 10 else 0\n        if sc > 0:\n            out[k, :len(w)] = np.clip((w - pre.mean()) / sc, -20, 20)\n    s = _seg(spo2, fs, a, a + WIN_PRE + WIN_POST)\n    if len(s):\n        s = np.where(np.isfinite(s), s, np.nan)\n        idx = np.linspace(0, len(s) - 1, n).astype(int)\n        s = s[idx]\n        pre = s[:WIN_PRE * NN_FS]\n        m = np.nanmedian(pre) if np.isfinite(pre).any() else np.nan\n        out[3] = np.nan_to_num((s - m) / 5.0, nan=0.0) if m == m else 0\n    i0 = WIN_PRE * NN_FS\n    out[4, i0:min(n, i0 + int(dur * NN_FS))] = 1.0\n    return out\n\n\n# ---------- UCDDB (from PhysioNet, the database the rule was built on) ----------\nUCDDB_HELDOUT = [\'ucddb003\', \'ucddb005\', \'ucddb007\', \'ucddb008\', \'ucddb010\', \'ucddb012\', \'ucddb015\',\n                 \'ucddb017\', \'ucddb020\', \'ucddb021\', \'ucddb023\', \'ucddb024\', \'ucddb026\', \'ucddb027\']\n\n\ndef edf_labels(path):\n    with open(path, \'rb\') as f:\n        h = f.read(256).decode(\'ascii\', \'ignore\'); ns = int(h[252:256])\n        return [f.read(16).decode(\'ascii\', \'ignore\').strip() for _ in range(ns)]\n\n\ndef pick(labels, *keys):\n    for lab in labels:\n        if any(k in lab.lower() for k in keys):\n            return lab\n    return None\n\n\n# ---------- evaluation ----------\ndef bal(p, l):\n    p, l = np.asarray(p, bool), np.asarray(l, bool)\n    return 0.5 * (p[l].mean() + (~p[~l]).mean())\n\n\ndef auc(s, l):\n    s, l = np.asarray(s, float), np.asarray(l, bool)\n    o = np.argsort(s, kind=\'mergesort\'); r = np.empty(len(s)); r[o] = np.arange(1, len(s) + 1)\n    for v in np.unique(s):\n        m = s == v\n        if m.sum() > 1: r[m] = r[m].mean()\n    n1 = l.sum(); n0 = len(l) - n1\n    return float((r[l].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))\n\n\ndef bootstrap_ci(score, y, groups, n=1000, seed=0):\n    """95% range of balanced accuracy, resampling whole patients/nights."""\n    rng = np.random.default_rng(seed); g = np.unique(groups); idx_of = {k: np.flatnonzero(groups == k) for k in g}\n    p = score > 0; out = []\n    for _ in range(n):\n        idx = np.concatenate([idx_of[k] for k in rng.choice(g, len(g))])\n        if y[idx].all() or (~y[idx]).all(): continue\n        out.append(bal(p[idx], y[idx]))\n    return np.percentile(out, [2.5, 97.5]) if out else (np.nan, np.nan)\n\n\ndef report(name, score, y, groups):\n    p = score > 0\n    lo, hi = bootstrap_ci(score, y, groups)\n    return {\'model\': name, \'events\': int(len(y)), \'balanced_acc\': 100 * bal(p, y), \'ci_low\': 100 * lo,\n            \'ci_high\': 100 * hi, \'auc\': auc(score, y), \'obstructive_caught\': 100 * p[y].mean(),\n            \'central_caught\': 100 * (~p[~y]).mean()}\n'
CHIPREF_PY = "import numpy as np\nfrom math import tan, pi, sqrt\nFS = 8\nf32 = np.float32\nGUARD_S, TAU_S, WARM_S = 20, 300.0, 60\nRH_MAX = 112\nSKIP_N, MIN_N = 24, 16\nSKIP_RH = SKIP_N // 2\nLAG_LO, LAG_HI = 8, 32\n\ndef _lp(fc):\n    K = tan(pi*fc/FS); n = 1/(1+sqrt(2)*K+K*K); b0 = K*K*n\n    return (b0, 2*b0, b0, 2*(K*K-1)*n, (1-sqrt(2)*K+K*K)*n)\ndef _hp(fc):\n    K = tan(pi*fc/FS); n = 1/(1+sqrt(2)*K+K*K)\n    return (n, -2*n, n, 2*(K*K-1)*n, (1-sqrt(2)*K+K*K)*n)\nCOEF = {'HP1': _hp(0.1), 'LP1': _lp(0.5), 'HP2': _hp(0.05), 'LP2': _lp(0.7)}\nC32 = {k: tuple(f32(c) for c in v) for k, v in COEF.items()}\n\ndef bq(c, z, i, x):\n    b0, b1, b2, a1, a2 = c\n    y = f32(f32(b0*x) + z[i])\n    z[i] = f32(f32(f32(b1*x) - f32(a1*y)) + z[i+1])\n    z[i+1] = f32(f32(b2*x) - f32(a2*y))\n    return y\n\nclass Ch:\n    def __init__(s):\n        s.z = [f32(0)]*4; s.sec = f32(0)\n        s.ring = [f32(0)]*GUARD_S; s.ok = [False]*GUARD_S\n        s.base = f32(0); s.nb = 0\n        s.s1 = s.s2 = s.sta = s.s1a = s.s2a = s.staa = f32(0); s.buf = []\n    def step(s, x):\n        return bq(C32['LP1'], s.z, 2, bq(C32['HP1'], s.z, 0, f32(x)))\n\ndef resid_ms(s1, s2, sta, n):\n    N = f32(n)\n    tm = f32(f32(N - f32(1)) * f32(0.5))\n    stt = f32(f32(N * f32(f32(N * N) - f32(1))) / f32(12))\n    saa = f32(s2 - f32(f32(s1 * s1) / N))\n    cta = f32(sta - f32(tm * s1))\n    r = f32(saa - f32(f32(cta * cta) / stt))\n    if r < 0: r = f32(0)\n    return f32(r / N)\n\ndef rhythm(buf):\n    n = len(buf)\n    hi = min(LAG_HI, n // 2)\n    if hi <= LAG_LO: return f32(0)\n    N = f32(n); tm = f32(f32(N - f32(1)) * f32(0.5))\n    m = f32(0)\n    for v in buf: m = f32(m + v)\n    m = f32(m / N)\n    st = f32(0); stt = f32(0)\n    for i, v in enumerate(buf):\n        t = f32(f32(i) - tm); st = f32(st + f32(t * f32(v - m))); stt = f32(stt + f32(t * t))\n    slope = f32(st / stt)\n    x = [f32(f32(v - m) - f32(slope * f32(f32(i) - tm))) for i, v in enumerate(buf)]\n    var = f32(0)\n    for v in x: var = f32(var + f32(v*v))\n    var = f32(var / N)\n    if var <= 0: return f32(0)\n    best = f32(-9)\n    for L in range(LAG_LO, hi + 1):\n        acc = f32(0)\n        for i in range(n - L): acc = f32(acc + f32(x[i]*x[i+L]))\n        c = f32(f32(acc / f32(n - L)) / var)\n        if c > best: best = c\n    return best\n\nclass Detector:\n    def __init__(s):\n        s.ch = [Ch(), Ch()]; s.k = 0; s.since = 10**6; s.inev = False\n        s.n = 0; s.pos = 0; s.b0 = [f32(0), f32(0)]; s.valid = False\n    def feed(s, rib, abd, ev):\n        out = None\n        ins = [f32(rib), f32(abd)]\n        a = [s.ch[0].step(rib), s.ch[1].step(abd)]\n        if ev and not s.inev:\n            s.inev = True; s.n = 0; s.b0 = [c.base for c in s.ch]\n            s.valid = s.ch[0].nb >= WARM_S and s.ch[1].nb >= WARM_S\n            for c in s.ch:\n                c.s1 = c.s2 = c.sta = c.s1a = c.s2a = c.staa = f32(0); c.buf = []; c.ok = [False]*GUARD_S\n        if s.inev and ev:\n            t = f32(s.n)\n            for c, x, v in zip(s.ch, a, ins):\n                c.s1a = f32(c.s1a + x); c.s2a = f32(c.s2a + f32(x*x)); c.staa = f32(c.staa + f32(t*x))\n                if s.n >= SKIP_N:\n                    u = f32(s.n - SKIP_N)\n                    c.s1 = f32(c.s1 + x); c.s2 = f32(c.s2 + f32(x*x)); c.sta = f32(c.sta + f32(u*x))\n                if s.n % 2 == 0 and len(c.buf) < RH_MAX: c.buf.append(v)\n            s.n += 1\n        if s.inev and not ev:\n            s.inev = False; s.since = 0\n            if s.valid and s.n > 0:\n                useskip = s.n >= SKIP_N + MIN_N\n                m = s.n - SKIP_N if useskip else s.n\n                amp = f32(0); rh = f32(-9)\n                for c, b in zip(s.ch, s.b0):\n                    ms = resid_ms(c.s1, c.s2, c.sta, m) if useskip else resid_ms(c.s1a, c.s2a, c.staa, s.n)\n                    r = f32(np.sqrt(f32(ms / b))) if b > 0 else f32(0)\n                    if r > amp: amp = r\n                    off = SKIP_RH if (useskip and len(c.buf) > SKIP_RH) else 0\n                    q = rhythm(c.buf[off:])\n                    if q > rh: rh = q\n                out = (float(amp), float(rh), s.n)\n        for c, x in zip(s.ch, a): c.sec = f32(c.sec + f32(x*x))\n        s.k += 1\n        if s.k % FS == 0:\n            clean = (not s.inev) and s.since >= GUARD_S\n            for c in s.ch:\n                old, ok = c.ring[s.pos], c.ok[s.pos]\n                if ok:\n                    if c.nb < WARM_S:\n                        c.nb += 1\n                        c.base = f32(c.base + f32(f32(old - c.base) / f32(c.nb)))\n                    else:\n                        if c.nb < 30000: c.nb += 1\n                        c.base = f32(c.base + f32(f32(old - c.base) / f32(TAU_S)))\n                c.ring[s.pos] = f32(c.sec / f32(FS)); c.ok[s.pos] = clean\n                c.sec = f32(0)\n            s.pos = (s.pos + 1) % GUARD_S\n            if not s.inev: s.since += 1\n        return out\n\ndef prepare(rib, abd, ev):\n    n = len(rib)\n    s = max(np.abs(rib).max(), np.abs(abd).max()) / 30000.0 or 1.0\n    ri = np.round(rib / s).astype(np.int16); ai = np.round(abd / s).astype(np.int16)\n    flag = np.zeros(n, np.uint8)\n    for t, d, ty in ev: flag[max(0, int(t*FS)):min(n, int((t+d)*FS))] = 1\n    return ri, ai, flag\n\ndef scored_events(ev, n):\n    out = []\n    for t, d, ty in ev:\n        if ty not in ('APNEA-O', 'APNEA-C'): continue\n        i0, i1 = int(t*FS), int((t+d)*FS)\n        if i0 < 40*FS or i1 + 12*FS > n or i1 - i0 < 4*FS: continue\n        out.append((i0, i1, ty[-1]))\n    return out\n\ndef match(results, scored):\n    res = sorted(results)\n    rows = []\n    for i0, i1, lab in scored:\n        cand = [r for r in res if r[0] >= i1 - 1 and r[0] - r[3] <= i0]\n        rows.append((i0, i1, lab, cand[0] if cand else None))\n    return rows\n"
UCDDB_IO_PY = "import re\nimport numpy as np\ndef read_edf(path, wanted):\n    with open(path, 'rb') as f:\n        f.read(8); f.read(80); f.read(80); f.read(8)\n        start = f.read(8).decode('ascii', 'ignore').strip()\n        f.read(8); f.read(44)\n        nrec = int(f.read(8).decode('ascii', 'ignore').strip())\n        rsec = float(f.read(8).decode('ascii', 'ignore').strip())\n        ns = int(f.read(4).decode('ascii', 'ignore').strip())\n        fld = lambda w: [f.read(w).decode('ascii', 'ignore').strip() for _ in range(ns)]\n        labels = fld(16); fld(80); fld(8)\n        pmin = [float(v) for v in fld(8)]; pmax = [float(v) for v in fld(8)]\n        dmin = [float(v) for v in fld(8)]; dmax = [float(v) for v in fld(8)]\n        fld(80)\n        spr = [int(v) for v in fld(8)]\n        f.read(32 * ns)\n        raw = f.read()\n    tot = sum(spr); bpr = tot * 2\n    nrec = min(nrec, len(raw) // bpr)\n    data = np.frombuffer(raw[:nrec * bpr], dtype='<i2').reshape(nrec, tot)\n    off = np.cumsum([0] + spr)\n    sig, rate = {}, {}\n    for i, lab in enumerate(labels):\n        key = 'Sound' if lab in ('Sound', 'Soud') else lab\n        if key not in wanted:\n            continue\n        blk = data[:, off[i]:off[i + 1]].ravel().astype(float)\n        span = (dmax[i] - dmin[i]) or 1.0\n        sig[key] = (blk - dmin[i]) * ((pmax[i] - pmin[i]) / span) + pmin[i]\n        rate[key] = spr[i] / rsec\n    p = re.split(r'[.:]', start)\n    h, m, s = (int(x) for x in p[:3]) if len(p) >= 3 else (0, 0, 0)\n    return sig, rate, h * 3600 + m * 60 + s\n\nEVENT_RE = re.compile(\n    r'^\\s*(\\d{1,2}):(\\d{2}):(\\d{2})\\s+([A-Z]+(?:-[A-Z])?)\\s+(?:(PB|CS)\\s+)?(\\d+)')\n\ndef parse_events(path, start_s):\n    out = []\n    with open(path, errors='ignore') as fh:\n        for line in fh:\n            m = EVENT_RE.match(line)\n            if m:\n                h, mi, s, typ, _, d = m.groups()\n                t = int(h) * 3600 + int(mi) * 60 + int(s) - start_s\n                if t < 0:\n                    t += 24 * 3600\n                out.append((t, int(d), typ.upper()))\n    return out\n\n"


## 2. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/SvasaLens_NN_v2'
os.makedirs(OUT + '/code', exist_ok=True)
for name, text in [('svnn.py', SVNN_PY), ('svexp.py', SVEXP_PY), ('chipref_v2.py', CHIPREF_PY), ('ucddb_io.py', UCDDB_IO_PY)]:
    open(f'{OUT}/code/{name}', 'w').write(text)
print('Ready.')


## 3. Settings
`SPLIT` and `SEED` decide how each hospital's patients are divided. Seed 0 keeps the same PhysioNet test patients as the first experiment.

In [ ]:
import os, sys, json, time, glob
import numpy as np
DRIVE = '/content/drive/MyDrive'
PN, PSG = f'{DRIVE}/PhysioNet2018', f'{DRIVE}/PSG-Audio'     # made by the two download notebooks
OUT = f'{DRIVE}/SvasaLens_NN_v2'                               # results of this notebook
UCD = '/content/ucddb'
SPLIT = (0.70, 0.10, 0.20)   # each hospital's patients: train / tuning / test (by patient, never mixed)
SEED = 0                     # same seed as the first experiment, so PhysioNet test patients are the same
USE_PSG_AUDIO = True         # all PSG-Audio patients are downloaded (286 of 287; 1 damaged on the server)
for d in (OUT, UCD): os.makedirs(d, exist_ok=True)
sys.path.insert(0, OUT + '/code')
import svnn as V, svexp as X, chipref_v2 as C, ucddb_io as U


## 4. Prepare apneas from PhysioNet 2018 and PSG-Audio

In [ ]:
def rule_rows(chip_file):
    """Scored apneas with the chip's effort and rhythm (same events the rule is judged on)."""
    z = np.load(chip_file)
    ev = [(float(t), float(d), str(ty)) for (t, d), ty in zip(z['events'], z['types'])]
    ri, ai, fl = z['ri'], z['ai'], z['fl']
    det, res = C.Detector(), []
    for i in range(len(ri)):
        o = det.feed(int(ri[i]), int(ai[i]), bool(fl[i]))
        if o: res.append((i, o[0], o[1], o[2]))
    return [(i0 / C.FS, (i1 - i0) / C.FS, lab, m[1], m[2])
            for i0, i1, lab, m in C.match(res, C.scored_events(ev, len(ri))) if m is not None]

def build(name, root, flow_key):
    files = sorted(glob.glob(f'{root}/chip_ready/*.npz'))
    cache = f'{OUT}/{name}_{len(files)}patients_events.npz'    # rebuilt automatically if the patient count changes
    if os.path.exists(cache):
        z = np.load(cache); return {k: z[k] for k in z.files}
    W, Y, G, E, R = [], [], [], [], []
    for k, cf in enumerate(files):
        pid = os.path.basename(cf)[:-4]
        jf = f'{root}/results/laptop_rule_per_patient/{pid}.json'
        rows = json.load(open(jf)) if os.path.exists(jf) else rule_rows(cf)
        if not rows: continue
        z = np.load(cf)
        chest, abd = V.chip_filter(z['ri'].astype(np.float32)), V.chip_filter(z['ai'].astype(np.float32))
        e = np.load(f'{root}/extracted/{pid}.npz')
        rate = float(e['rate']) if 'rate' in e.files and e['rate'].ndim == 0 and e['rate'].dtype.kind in 'fi' \
            else json.loads(str(e['rates']))[flow_key]
        flow = V.flow_filter(X.resample(e[flow_key], rate, V.FS))
        for t0, dur, lab, eff, rh in rows:
            W.append(V.window([chest, abd, flow], t0, dur).astype(np.float16))
            Y.append(lab == 'O'); G.append(pid); E.append(eff); R.append(rh)
        print(f'\r  {name}: {k+1}/{len(files)} patients', end='')
    d = {'W': np.stack(W), 'Y': np.array(Y), 'G': np.array(G), 'E': np.array(E, np.float32), 'R': np.array(R, np.float32)}
    np.savez(cache, **d)
    return d

DATA = {'PhysioNet 2018': build('pn18', PN, 'airflow')}
def _psg_problem():
    if not USE_PSG_AUDIO: return 'USE_PSG_AUDIO is False (section 2)'
    if not os.path.isdir(PSG):
        return (f'no folder {PSG}. Folders in My Drive: {sorted(os.listdir(DRIVE))[:30]}')
    for sub in ('chip_ready', 'extracted'):
        if not os.path.isdir(f'{PSG}/{sub}'):
            return f'no folder {PSG}/{sub}. Inside {PSG}: {sorted(os.listdir(PSG))[:30]}'
    n_chip = len(glob.glob(f'{PSG}/chip_ready/*.npz')); n_ext = len(glob.glob(f'{PSG}/extracted/*.npz'))
    if n_chip == 0: return f'{PSG}/chip_ready has no .npz files. Inside it: {sorted(os.listdir(f"{PSG}/chip_ready"))[:10]}'
    missing = [os.path.basename(f) for f in glob.glob(f'{PSG}/chip_ready/*.npz') if not os.path.exists(f'{PSG}/extracted/' + os.path.basename(f))]
    if missing: return f'{len(missing)} patients have chip files but no extracted file (upload not finished?), e.g. {missing[:5]}'
    print(f'PSG-Audio found: {n_chip} chip files, {n_ext} extracted files')
    return None

problem = _psg_problem()
if problem is None:
    DATA['PSG-Audio'] = build('psgaudio', PSG, 'thermistor')
else:
    print(f'PSG-Audio LEFT OUT: {problem}. Training on PhysioNet 2018 only.')
for k, d in DATA.items():
    print(f'\n{k}: {len(d["Y"])} apneas from {len(np.unique(d["G"]))} patients '
          f'(obstructive {d["Y"].sum()}, central {(~d["Y"]).sum()})')


## 5. Prepare UCDDB (outside test)

In [ ]:
import requests
cache = f'{OUT}/ucddb_events.npz'
if os.path.exists(cache):
    z = np.load(cache); UD = {k: z[k] for k in z.files}
else:
    nights = [f'ucddb{n:03d}' for n in [2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]]
    def get(name):
        dest = f'{UCD}/{name}'
        if not os.path.exists(dest):
            for base in ('https://physionet-open.s3.amazonaws.com/ucddb/1.0.0/', 'https://physionet.org/files/ucddb/1.0.0/'):
                r = requests.get(base + name, timeout=120)
                if r.status_code == 200: open(dest, 'wb').write(r.content); break
        return dest
    W, Y, G, E, R = [], [], [], [], []
    for k, sid in enumerate(nights):
        rec, evf = get(f'{sid}.rec'), get(f'{sid}_respevt.txt')
        fl_lab = X.pick(X.edf_labels(rec), 'flow')
        sig, rate, st = U.read_edf(rec, {'ribcage', 'abdo', fl_lab})
        ev = U.parse_events(evf, st)
        n = min(len(sig['ribcage']), len(sig['abdo']))
        ri, ai, fl = C.prepare(sig['ribcage'][:n], sig['abdo'][:n], ev)
        det, res = C.Detector(), []
        for i in range(len(ri)):
            o = det.feed(int(ri[i]), int(ai[i]), bool(fl[i]))
            if o: res.append((i, o[0], o[1], o[2]))
        chest, abd = V.chip_filter(ri.astype(np.float32)), V.chip_filter(ai.astype(np.float32))
        flow = V.flow_filter(X.resample(sig[fl_lab], rate[fl_lab], V.FS))
        for i0, i1, lab, m in C.match(res, C.scored_events(ev, len(ri))):
            if m is None: continue
            W.append(V.window([chest, abd, flow], i0 / C.FS, (i1 - i0) / C.FS).astype(np.float16))
            Y.append(lab == 'O'); G.append(sid); E.append(m[1]); R.append(m[2])
        print(f'\r  UCDDB: {k+1}/{len(nights)} nights', end='')
    UD = {'W': np.stack(W), 'Y': np.array(Y), 'G': np.array(G), 'E': np.array(E, np.float32), 'R': np.array(R, np.float32)}
    np.savez(cache, **UD)
print(f'\nUCDDB: {len(UD["Y"])} apneas from {len(np.unique(UD["G"]))} nights')


## 6. Split patients

In [ ]:
def split(G):
    rng = np.random.default_rng(SEED)
    pats = np.array(sorted(np.unique(G))); rng.shuffle(pats)
    a, b = int(SPLIT[0] * len(pats)), int(SPLIT[0] * len(pats)) + int(SPLIT[1] * len(pats))
    return np.isin(G, pats[:a]), np.isin(G, pats[a:b]), np.isin(G, pats[b:])
S = {k: split(d['G']) for k, d in DATA.items()}
for k, (tr, va, te) in S.items():
    print(f'{k}: apneas train {tr.sum()}, tuning {va.sum()}, test {te.sum()}')
held = np.isin(UD['G'], X.UCDDB_HELDOUT)
def sub(d, m): return {k: v[m] for k, v in d.items()}
TESTS = {f'{k} test patients': sub(d, S[k][2]) for k, d in DATA.items()}
if 'PSG-Audio' in DATA:
    TESTS['PSG-Audio, all patients (only fair for models not trained on it)'] = DATA['PSG-Audio']
TESTS['UCDDB, 14 held-out nights'] = sub(UD, held)
TESTS['UCDDB, all 25 nights'] = UD


## 7. Train the networks

In [ ]:
import torch, torch.nn as nn
REUSE_TRAINED = True    # True = models already trained (saved in Drive) are loaded, not trained again
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
class Net(nn.Module):
    def __init__(s, ch):
        super().__init__()
        s.f = nn.Sequential(nn.Conv1d(ch, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
                            nn.Conv1d(16, 32, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
                            nn.Conv1d(32, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        s.d = nn.Dropout(0.3); s.o = nn.Linear(32, 1)
    def forward(s, x): return s.o(s.d(s.f(x).squeeze(-1))).squeeze(-1)

def predict(net, W, ch):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), 2048):
            out.append(net(torch.tensor(W[i:i + 2048][:, ch].astype(np.float32), device=dev)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

def model_file(name):
    return f"{OUT}/{name.replace(' ', '_').replace(',', '').replace('+', 'plus')}.pt"

def train(name, sources, ch):
    if REUSE_TRAINED and os.path.exists(model_file(name)):        # already trained in an earlier run: load it
        ck = torch.load(model_file(name), map_location=dev)
        net = Net(len(ch)).to(dev); net.load_state_dict(ck['weights']); net.cut = ck['cut']
        print(f'{name}: loaded the model trained earlier'); return net
    Xtr = np.concatenate([DATA[s]['W'][S[s][0]][:, ch] for s in sources])
    Ytr = np.concatenate([DATA[s]['Y'][S[s][0]] for s in sources]).astype(np.float32)
    tune = [(DATA[s]['W'][S[s][1]], DATA[s]['Y'][S[s][1]]) for s in sources]
    torch.manual_seed(SEED); rng = np.random.default_rng(SEED)
    net = Net(len(ch)).to(dev)
    opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1 - Ytr).sum() / max(Ytr.sum(), 1), device=dev))
    print(f'{name}: {sum(p.numel() for p in net.parameters())} weights, {len(Ytr)} training apneas')
    best, state, bad = -1, None, 0
    for ep in range(40):
        net.train(); perm = rng.permutation(len(Xtr))
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            xb = torch.tensor(V.augment(Xtr[b].astype(np.float32), rng), device=dev)
            opt.zero_grad(); lossf(net(xb), torch.tensor(Ytr[b], device=dev)).backward(); opt.step()
        vs = [X.auc(predict(net, w, ch), y) for w, y in tune if y.any() and (~y).any()]
        v = float(np.mean(vs)) if vs else 0.0                               # average over hospitals
        print(f'  round {ep+1}: tuning AUC {v:.3f}')
        if state is None or v > best: best, state, bad = v, {k: t.clone() for k, t in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 5: break
    net.load_state_dict(state)
    # cut-off between obstructive and central, chosen on the tuning patients only
    tt = [(predict(net, w, ch), y) for w, y in tune if y.any() and (~y).any()]
    if tt:
        cands = np.quantile(np.concatenate([s for s, _ in tt]), np.linspace(0.02, 0.98, 97))
        net.cut = float(max(cands, key=lambda c: np.mean([X.bal(s > c, y) for s, y in tt])))
    else:
        net.cut = 0.0
    print(f'  cut-off set on tuning patients: {net.cut:.3f}')
    torch.save({'weights': state, 'cut': net.cut, 'channels': ch, 'sources': sources}, model_file(name))
    return net

results, PRED = [], {}
def add(name, scorer, trained_on):
    for tname, d in TESTS.items():
        if tname.startswith('PSG-Audio, all') and 'PSG-Audio' in trained_on: continue
        sc = scorer(d)
        PRED[(name, tname)] = (sc, d['Y'], d['G'])          # kept for the per-patient results (section 9)
        r = X.report(name, sc, d['Y'], d['G']); r['test'] = tname; results.append(r)

add('Original rule (UCDDB, unchanged)', lambda d: X.RULE[0] + X.RULE[1] * np.log(np.maximum(d['E'], 1e-3)) + X.RULE[2] * d['R'], [])
EFF, EFF_FLOW = [0, 1, 3], [0, 1, 2, 3]        # rows: chest, belly, airflow, event marker
plan = [('NN v2, effort, PhysioNet only', ['PhysioNet 2018'], EFF),
        ('NN v2, effort + airflow, PhysioNet only', ['PhysioNet 2018'], EFF_FLOW)]
if 'PSG-Audio' in DATA:
    plan += [('NN v2, effort, two hospitals', ['PhysioNet 2018', 'PSG-Audio'], EFF),
             ('NN v2, effort + airflow, two hospitals', ['PhysioNet 2018', 'PSG-Audio'], EFF_FLOW)]
for name, src, ch in plan:
    net = train(name, src, ch)
    add(name, lambda d, n=net, c=ch: predict(n, d['W'], c) - n.cut, src)


## 8. Results

In [ ]:
import csv
L = [f'SvasaLens neural network v2  ({time.strftime("%Y-%m-%d %H:%M")}, seed {SEED})',
     'Chip-style filtered input, random sign/size/noise/drift/shift during training.',
     'UCDDB and each hospital\'s test patients were never used for training or tuning.', '']
for tname in TESTS:
    rs = [r for r in results if r['test'] == tname]
    if not rs: continue
    L.append(tname)
    L.append(f"  {'model':40s} {'balanced':>9s} {'95% range':>15s} {'AUC':>6s} {'obstr.':>7s} {'central':>8s}")
    for r in rs:
        L.append(f"  {r['model']:40s} {r['balanced_acc']:8.1f}% {r['ci_low']:6.1f}-{r['ci_high']:5.1f}% "
                 f"{r['auc']:6.3f} {r['obstructive_caught']:6.1f}% {r['central_caught']:7.1f}%")
    L.append('')
L.append('For the original rule, only the 14 held-out UCDDB nights are fair (it was built on the other 11).')
print('\n'.join(L))
open(f'{OUT}/nn_v2_results.txt', 'w').write('\n'.join(L) + '\n')
with open(f'{OUT}/nn_v2_results.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, results[0].keys()); w.writeheader(); w.writerows(results)


## 9. Per-patient results
Doctors choose treatment per patient, so this section judges whole patients. The rules were fixed **before** seeing any results:
- A patient is **central-predominant** if **more than 50%** of their scored apneas are central (the usual clinical cut-off). The same counting is applied to the technologist's labels and to each model's calls.
- Only patients with **at least 5 scored apneas** are judged.

"Central found" is the share of truly central-predominant patients the model also called central-predominant; "obstructive found" is the same for obstructive-predominant patients.

In [ ]:
import csv
MIN_APNEAS, CENTRAL_SHARE = 5, 0.5      # fixed before looking at results

def wilson(k, n, z=1.96):
    """95% range for a proportion k/n (Wilson method; works for small n)."""
    if n == 0: return float('nan'), float('nan')
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, 100 * (c - h)), min(100.0, 100 * (c + h))

def per_patient(sc, Y, G):
    rows = []
    for g in np.unique(G):
        m = G == g
        if m.sum() < MIN_APNEAS: continue
        rows.append((np.mean(~Y[m]) > CENTRAL_SHARE, np.mean(sc[m] <= 0) > CENTRAL_SHARE))
    t = np.array([r[0] for r in rows], bool); p = np.array([r[1] for r in rows], bool)
    return t, p

PP = ['', 'PER-PATIENT RESULTS (central-predominant = more than 50% of scored apneas central; at least 5 apneas)']
pp_rows = []
for tname in TESTS:
    ms = [m for (m, t) in PRED if t == tname]
    if not ms: continue
    t0, _ = per_patient(*PRED[(ms[0], tname)])
    PP.append(f'{tname}: {len(t0)} patients judged ({t0.sum()} central-predominant, {(~t0).sum()} obstructive-predominant)')
    PP.append(f"  {'model':40s} {'correct':>8s} {'central found':>22s} {'obstructive found':>24s}")
    for m in ms:
        t, p = per_patient(*PRED[(m, tname)])
        kc, nc = int((p & t).sum()), int(t.sum()); ko, no = int((~p & ~t).sum()), int((~t).sum())
        cl, ch = wilson(kc, nc); ol, oh = wilson(ko, no)
        cen = f'{kc}/{nc} ({100*kc/nc:.0f}%, {cl:.0f}-{ch:.0f})' if nc else 'none'
        obs = f'{ko}/{no} ({100*ko/no:.0f}%, {ol:.0f}-{oh:.0f})' if no else 'none'
        PP.append(f"  {m:40s} {100*np.mean(p == t):7.1f}% {cen:>22s} {obs:>24s}")
        pp_rows.append({'test': tname, 'model': m, 'patients': len(t), 'correct_pct': 100 * np.mean(p == t),
                        'central_found': kc, 'central_patients': nc, 'obstructive_found': ko, 'obstructive_patients': no})
    PP.append('')
PP.append('Ranges in brackets are 95% ranges. With few central-predominant patients they are wide: read them, not just the %.')
print('\n'.join(PP))
open(f'{OUT}/nn_v2_results.txt', 'a').write('\n'.join(PP) + '\n')
with open(f'{OUT}/nn_v2_per_patient.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, pp_rows[0].keys()); w.writeheader(); w.writerows(pp_rows)
